# Before and after: feathers as rows, labels read off the slide, per-feather checks

Compares the April pipeline run (**before**, Drive `runs/20260426-075440`) with a run of the current pipeline
(**after**) on the same slides.

| Goal | Before | After |
|---|---|---|
| **1. Feathers as rows, metadata off the slide** | one row per slide (`image_segmentation_qc_*.csv`); bird ID and year from the file name; no species, age, sex or full date; crops named after the slide | one row per feather (`feathers.csv`) with its feather ID; bird ID, species, age, sex and date read off the slide label, each with a status; duplicates across same-day slides marked |
| **2. Per-feather scores** | one 1-10 VLM score per slide plus slide-wide flags (grouped boxes, leakage, all covered) | each feather checked (outline per side, holes, single feather, fragment, other feather or material inside), a score computed from the checks, failures confirmed on a second look; fragments set aside; `usable` per feather |

The before crops were never checked one by one. To compare segmentation feather by feather, section 2 can score
them with today's checks (`scripts/judge_existing_crops.py`, same prompt and model, via the VLM server).

Kernel: **bird-feather-analysis (pixi)**. Images open full size in a new JupyterLab tab when clicked.

## 1. Parameters

In [ ]:
from pathlib import Path

BEFORE_RUN = "20260426-075440"                       # Drive: Feather_Molt_Project/runs/<run>/
BEFORE_DIR = Path("../data/before_run")              # local copy: the QC table, crops/, overlays/
BEFORE_CHECKS = BEFORE_DIR / "feather_checks.csv"    # today's checks on the before crops (section 2c)
AFTER_RUN = "slice100-final-20261020"                 # a run of the current pipeline (scripts/run_slice.py);
                                                     # while it is still running, the newest finished run is used
GOLD = "../data/label_gold.csv"                      # hand-checked answers (30 slides)
DRIVE = "gdrive:Feather_Molt_Project/runs"

FETCH_BEFORE = True        # download the before table and images for these slides from Drive when missing
SCORE_BEFORE = False       # run today's checks on the before crops (about 15 s per feather, VLM server)
GRID_ORDER = "change"      # slide grid: "change" (most improved and worst first) | "manifest"
GRID_ONLY_GOLD = False     # True: only the hand-checked slides

## 2. Setup and load

In [ ]:
import base64, glob, html, io, json, subprocess, sys
from collections import Counter
from urllib.parse import quote
sys.path.insert(0, "..")
import pandas as pd
from IPython.display import display, HTML, Markdown
from PIL import Image
from src import label_extract as lx, output_names as on

pd.set_option("display.max_colwidth", 70)

def _finished(run):
    return all((Path("../data/label_runs") / run / f).exists() for f in ("manifest.txt", "labels.jsonl", "feathers.csv"))

if not _finished(AFTER_RUN):
    done = sorted((p for p in Path("../data/label_runs").glob("slice*") if _finished(p.name)),
                  key=lambda p: (p / "summary.json").stat().st_mtime if (p / "summary.json").exists() else 0)
    if not done:
        raise SystemExit(f"{AFTER_RUN} has no tables yet (still running?) and no other finished run was found")
    print(f"{AFTER_RUN} is not finished (its tables appear in data/label_runs/ when the run ends); "
          f"using the newest finished run, {done[-1].name}")
    AFTER_RUN = done[-1].name
AFTER_LABELS = Path("../data/label_runs") / AFTER_RUN   # labels.jsonl, feathers.csv, summary.json
AFTER_SEG = Path("../data/sample_runs") / AFTER_RUN     # crops, overlays, feather_qc/
names = [l.strip() for l in (AFTER_LABELS / "manifest.txt").read_text().splitlines() if l.strip()]
stems = [Path(n).stem for n in names]

def fetch_before():
    """The before run's QC table, and crops + overlays of these slides only, from Drive. Overlays sit in
    'Bounding Boxes' and among the crops in 'All feathers'. Slides left out of the share
    (include_in_share = 0) mostly have no crops there."""
    BEFORE_DIR.mkdir(parents=True, exist_ok=True)
    src = f"{DRIVE}/{BEFORE_RUN}"
    subprocess.run(["rclone", "copy", f"{src}/image_segmentation_qc_{BEFORE_RUN}.csv", str(BEFORE_DIR), "-q"], check=True)
    inc = BEFORE_DIR / "include.txt"
    inc.write_text("".join(glob.escape(s) + "_*\n" for s in stems))
    for remote, local in (("All feathers", "crops"), ("Bounding Boxes", "overlays")):
        subprocess.run(["rclone", "copy", f"{src}/{remote}", str(BEFORE_DIR / local), "--include-from", str(inc),
                        "--transfers", "16", "-q"], check=True)
    for p in (BEFORE_DIR / "crops").glob("*_BoundingBoxes.jpg"):
        p.replace(BEFORE_DIR / "overlays" / p.name)

before_csv = BEFORE_DIR / f"image_segmentation_qc_{BEFORE_RUN}.csv"
if FETCH_BEFORE and not (before_csv.exists() and (BEFORE_DIR / "overlays").exists()):
    fetch_before()
before = pd.read_csv(before_csv)
before["file_id"] = before.image_path.map(lambda p: Path(p).name)
before = before[before.file_id.isin(names)].set_index("file_id").reindex(names)

def before_crops(stem):
    """Crops of one slide in the before run (old names '<slide>_<id>_<year>_Feather_<n>.jpg'), by n."""
    found = (BEFORE_DIR / "crops").glob(glob.escape(stem) + "_*_Feather_*.jpg")
    return sorted(found, key=lambda p: int(p.stem.rsplit("_", 1)[-1]))

latest = lx.load_previous(AFTER_LABELS / "labels.jsonl")
results = [latest[n] for n in names if n in latest]
after_summary = json.loads((AFTER_LABELS / "summary.json").read_text())
fdf = pd.read_csv(AFTER_LABELS / "feathers.csv")
for c in ("needs_review", "review_uncertain", "duplicate_of", "feather_id", "segment_status"):
    fdf[c] = fdf[c].fillna("") if c in fdf else ""
if "usable" not in fdf:     # runs made before the usable column existed
    fdf["usable"], fdf["excluded_because"] = zip(*(("yes" if ok else "no", why)
                                                    for ok, why in (lx.usable(f) for f in fdf.to_dict("records"))))
dup = {(r.file_id, r.seq): r.duplicate_of for r in fdf.itertuples()}
for r in results:
    for f in r.feathers:
        f["duplicate_of"] = dup.get((r.file_id, f.get("seq")), "")
gold = lx.load_gold(GOLD) if Path(GOLD).exists() else {}
print(f"{len(names)} slides | before: {before.feathers_saved.notna().sum()} slide rows, "
      f"{sum(len(before_crops(s)) for s in stems)} crops | after: {len(results)} slides, {len(fdf)} feather-table rows")

## 3. Goal 1: feathers as rows, metadata read off the slide
### 3a. The table, before and after
The same three slides. Before: one row per slide, nothing per feather. After: one row per feather with its ID and the
slide's label fields repeated (plus fragments and review rows, which have no feather ID).

In [ ]:
EXAMPLES = names[:3]
display(Markdown(f"**Before** (`image_segmentation_qc_{BEFORE_RUN}.csv`): {len(before)} rows for {len(names)} slides"))
display(before.loc[EXAMPLES, ["bird_id", "date_text", "metadata_source", "feathers_saved", "vlm_score", "vlm_notes"]])
cols = ["file_id", "feather_id", "seq", "segment_status", "usable", "bird_id", "species", "age", "sex", "year",
        "month", "day", "judge_score", "needs_review", "crop"]
display(Markdown(f"**After** (`feathers.csv`): {len(fdf)} rows for {fdf.file_id.nunique()} slides"))
display(fdf[fdf.file_id.isin(EXAMPLES)][[c for c in cols if c in fdf]])

### 3b. Metadata: file name vs slide label
Before, only the bird ID and the year existed, both from the file name (`A0012 1996-im1058.jpg` -> `A0012`, `1996`).
The label's ID carries a suffix letter (`A0012H`), and the rest of the fields were not recorded. Right = matches the
hand-checked answer on the slides that have one.

In [ ]:
def _gold(field):
    return {n: (gold.get(n) or {}).get(field, "").strip() for n in names if (gold.get(n) or {}).get(field, "").strip()}

def _right(got, want):
    return sum(str(got.get(n) or "").strip().upper() == w.upper() for n, w in want.items())

rows = []
after_gold = after_summary.get("gold", {})
readable = after_summary.get("readable_rate", {})
for field in ("bird_id", "species", "age", "sex", "date"):
    want = _gold(field)
    if field == "bird_id":
        b_got = before.bird_id.to_dict()
    elif field == "date":
        b_got = {n: str(int(float(v))) if pd.notna(v) else "" for n, v in before.date_text.items()}
        want = {n: w[:4] for n, w in want.items()}            # before had the year only
    else:
        b_got = {}
    g = after_gold.get(field, {})
    rows.append({"field": field,
                 "before: source": "file name" if b_got else "not recorded",
                 "before: right": (f"{_right(b_got, want)}/{len(want)}" + (" (year only)" if field == "date" else ""))
                                  if b_got else "-",
                 "after: source": "slide label (file name as backup)",
                 "after: readable": f"{readable.get(field, 0):.0%}",
                 "after: right": f"{g.get('correct', '-')}/{g.get('n', '-')}"})
display(pd.DataFrame(rows).set_index("field"))
print("after, status counts per field:")
display(pd.DataFrame(after_summary["status_counts"]).fillna(0).astype(int).T)

### 3c. Feather IDs
Before there were no feather IDs; a crop was known by its slide's file name and a number. An ID from the file name's
bird and year alone (`<id>_<year>_<n>`) is not unique: a bird with several slides in one year repeats it. After, the ID
is `<bird_id>_<period>_<letter>`, with the label date (and image number) when the bird has several slides that year,
and repeats of the same feather on same-day slides are marked `duplicate_of`.

In [ ]:
naive = Counter()
for s in stems:
    for p in before_crops(s):
        bird, year, n = p.name.split("_")[-4], p.name.split("_")[-3], p.stem.rsplit("_", 1)[-1]
        naive[f"{bird}_{year}_{n}"] += 1
clash = {k: v for k, v in naive.items() if v > 1}
seg = fdf[fdf.segment_status == "segmented"]
display(pd.DataFrame([
    {"": "before", "feathers": sum(naive.values()), "IDs": "none (file name + number)",
     "would clash as <id>_<year>_<n>": sum(clash.values()), "duplicates marked": "-"},
    {"": "after", "feathers": len(seg), "IDs": f"{seg.feather_id.nunique()} unique",
     "would clash as <id>_<year>_<n>": 0, "duplicates marked": int((seg.duplicate_of.fillna("") != "").sum())},
]).set_index(""))

## 4. Goal 2: per-feather scores
### 4a. What each run says about segmentation quality
Before: one score per slide, and flags for the whole slide. Which feather was the problem is not recorded.

In [ ]:
b = before
print(f"before: slide score counts (1-10), {len(b)} slides:")
display(b.vlm_score.value_counts().sort_index().to_frame("slides").T)
flags = {"grouped boxes detected": (b.vlm_grouped_boxes_detected == 1).sum(),
         "background leakage": (b.vlm_background_leakage_detected == 1).sum(),
         "not all feathers covered": (b.vlm_all_feathers_covered == 0).sum(),
         "excluded from the share (include_in_share = 0)": (b.include_in_share == 0).sum()}
display(pd.Series(flags, name="slides").to_frame().T)

s = fdf[fdf.segment_status == "segmented"]
kinds = Counter(k.strip().split(":")[0] for v in s.needs_review.fillna("") for k in v.split(";") if k.strip())
print(f"\nafter: {len(s)} feathers checked one by one; feather score counts (computed from the checks):")
display(s.judge_score.round().value_counts().sort_index().to_frame("feathers").T)
display(pd.Series({
    "usable for analysis": int((fdf.usable == "yes").sum()),
    "failed a check (confirmed)": int((s.needs_review.fillna("") != "").sum()),
    "failed on one look only (listed, still usable)": int((s.review_uncertain.fillna("") != "").sum()),
    "duplicates": int((s.duplicate_of.fillna("") != "").sum()),
    "fragments set aside": int((fdf.segment_status == "fragment").sum()),
    "possible missed feathers (review rows)": int((fdf.segment_status == "possible_missed").sum()),
}, name="after").to_frame().T)
print("after, failed checks by kind:")
display(pd.Series(kinds, name="feathers").sort_values(ascending=False).to_frame().T)

### 4b. Feather counts against the hand counts
Feathers kept per slide vs the number on the slide, on the hand-checked slides.

In [ ]:
after_n = {r.file_id: len(lx.segmented(r.feathers)) for r in results}
cmp = pd.DataFrame([{"file_id": n, "on the slide": int(gold[n]["feather_count"]),
                     "before": int(before.feathers_saved.get(n) or 0), "after": after_n.get(n)}
                    for n in names if (gold.get(n) or {}).get("feather_count", "").strip()]).set_index("file_id")
print(f"right count: before {(cmp.before == cmp['on the slide']).sum()}/{len(cmp)}, "
      f"after {(cmp.after == cmp['on the slide']).sum()}/{len(cmp)}")
diff = cmp[(cmp.before != cmp["on the slide"]) | (cmp.after != cmp["on the slide"])]
display(diff if len(diff) else Markdown("_all counts right in both runs_"))

### 4c. The same checks on the before crops
Scores the before crops with today's per-feather checks (same prompt, model and second look), so the segmentation of
both runs is judged the same way. Each crop is located on its slide to rebuild its mask. Set `SCORE_BEFORE = True`
to run it (or run `scripts/judge_existing_crops.py` from a terminal); results are kept in `BEFORE_CHECKS`.

Two limits: the measured **hidden** check is left out on both sides (the before masks come from the older segmenter,
whose smoother outlines it misreads), and the model **cannot see glue**, the most common fault in your hand verdicts.
So fewer failed checks here does not mean cleaner crops; section 4d measures the glue directly.

In [ ]:
if SCORE_BEFORE and not BEFORE_CHECKS.exists():
    subprocess.run([sys.executable, "../scripts/judge_existing_crops.py", str(BEFORE_DIR / "crops"), str(BEFORE_CHECKS),
                    "--slides", "../data/raw_cache", "--manifest", str(AFTER_LABELS / "manifest.txt")], check=True)
bchk = pd.read_csv(BEFORE_CHECKS) if BEFORE_CHECKS.exists() else None
if bchk is not None:
    bchk["needs_review"] = bchk.needs_review.fillna("")
    after_seg = fdf[fdf.segment_status == "segmented"].copy()
    after_seg["needs_review"] = after_seg.needs_review.fillna("")
    def _kinds(col):
        return Counter(k.strip().split(":")[0] for v in col for k in v.split(";") if k.strip())
    after_seg["needs_review"] = after_seg.needs_review.map(
        lambda v: "; ".join(i for i in v.split("; ") if i and not i.endswith("hidden")))   # like the before side
    kb, ka = _kinds(bchk.needs_review), _kinds(after_seg.needs_review)
    display(pd.DataFrame({
        "before": {"feathers": len(bchk), "failed a check": int((bchk.needs_review != "").sum()),
                   "mean score": round(bchk.judge_score.mean(), 2), **kb},
        "after": {"feathers": len(after_seg), "failed a check": int((after_seg.needs_review != "").sum()),
                  "mean score": round(after_seg.judge_score.mean(), 2), **ka}}).fillna(0))
    per = pd.DataFrame({"before": bchk.groupby("file_id").needs_review.apply(lambda c: (c != "").sum()),
                        "after": after_seg.groupby("file_id").needs_review.apply(lambda c: (c != "").sum())}).fillna(0)
    print(f"slides with fewer failing feathers after: {(per.after < per.before).sum()}, "
          f"same: {(per.after == per.before).sum()}, more: {(per.after > per.before).sum()}")
    print("(the checks vary a little between runs on the same mask: compare whole groups, not single flags)")
else:
    print("no checks on the before crops yet: set SCORE_BEFORE = True (or run scripts/judge_existing_crops.py)")

### 4d. Glue and card left in the saved crops (measured)
Glue at the quill base and card inside the crop were the most common faults in your hand verdicts, and the model does
not see them. Measured instead on every saved crop of both runs: patches without barb texture that touch the crop's
edge (the detector of `src/mask_cleanup.py`, which found all 8 glue cases among the hand-checked feathers and no
good feather). Computed once, then cached next to the before files.

In [ ]:
import cv2
from src import mask_cleanup
from src.crop_locate import crop_mask
from src.feather_processing import load_seg_config, make_feather

GLUE_CACHE = BEFORE_DIR / f"glue_left_{AFTER_RUN}.csv"
if not GLUE_CACHE.exists():
    seg_cfg = load_seg_config()
    ccfg, trim = seg_cfg["crop"]["cleanup"], float(seg_cfg["crop"].get("trim_bottom_frac", 0.12))
    rows = []
    for n in names:
        img = cv2.imread(str(Path("../data/raw_cache") / n))
        for p in before_crops(Path(n).stem):
            m, _ = crop_mask(img, cv2.imread(str(p)))
            rows.append({"run": "before", "file_id": n, "crop": p.name,
                         "glue_patches": len(mask_cleanup.textureless_patches(img, m, ccfg))})
        for r in fdf[(fdf.file_id == n) & (fdf.segment_status == "segmented")].itertuples():
            if not isinstance(r.mask_image, str):
                continue
            raw = cv2.imread(str(AFTER_SEG / r.mask_image), 0) > 0
            m = make_feather(img, raw, trim, 0, {**ccfg, "enabled": True})["mask"]     # the saved crop's mask
            rows.append({"run": "after", "file_id": n, "crop": r.crop,
                         "glue_patches": len(mask_cleanup.textureless_patches(img, m, ccfg))})
    pd.DataFrame(rows).to_csv(GLUE_CACHE, index=False)
glue = pd.read_csv(GLUE_CACHE)
glue_of = {(r.run, r.crop): r.glue_patches for r in glue.itertuples()}
display(glue.assign(with_glue=glue.glue_patches > 0).groupby("run").agg(
    crops=("crop", "size"), crops_with_glue_or_card=("with_glue", "sum")).T)

## 5. Slide grid: before | after
One card per slide: the before run on the left, the after run on the right. **Click any image** to open it full size.

- **Before:** the overlay, the crops numbered as saved, the slide's one score and notes, the bird ID and year from
  the file name. With section 4c run, each crop also shows today's score, outlined orange when it fails a check.
- **After:** the overlay, the feathers lettered left to right with their computed score. Outlines: green = usable,
  orange = failed a check, red = duplicate, grey dashed = fragment (no ID), red dashed box = possible missed feather.
  Label fields in status colours, feather IDs below.
- **Card border:** green = after is better (feather count fixed, or fewer feathers that fail a check or keep glue/card
  in the crop; hidden flags left out on both sides), red = after is worse, grey = no clear change.

In [ ]:
STATUS_COLORS = {"readable": "#d4edda", "partial": "#fff3cd", "not_found": "#e2e3e5", "filename_fallback": "#fff3cd",
                 "needs_review": "#f8d7da", "conflict": "#f8d7da", "model_error": "#f5c6cb"}
bscore = ({r.crop: r for r in bchk.itertuples()} if bchk is not None else {})

def _thumb(path, size):
    im = Image.open(path).convert("RGB"); im.thumbnail(size)
    buf = io.BytesIO(); im.save(buf, "JPEG", quality=80)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

def _rel(path):
    return quote(str(Path("..") / Path(path).resolve().relative_to(Path("..").resolve())))

def _img(path, size, style=""):
    return (f'<a href="{_rel(path)}" target="_blank"><img src="{_thumb(path, size)}" style="{style}"></a>'
            if path and Path(path).exists() else "<i>missing</i>")

def _crop(path, label, color, tip, dashed=False):
    return (f'<div style="display:inline-block;text-align:center;margin:1px" title="{html.escape(tip)}">'
            + _img(path, (60, 140), f"height:70px;outline:3px {'dashed' if dashed else 'solid'} {color}")
            + f'<br><span style="font-size:10px">{html.escape(label)}</span></div>')

def _year(v):
    return str(int(v)) if isinstance(v, (int, float)) and not pd.isna(v) else str(v or "-")

def _before_side(n):
    stem, row = Path(n).stem, before.loc[n]
    crops = []
    for p in before_crops(stem):
        k = bscore.get(p.name)
        sc = "" if k is None else f" {k.judge_score:g}"
        bad = k is not None and str(k.needs_review or "") not in ("", "nan")
        crops.append(_crop(p, p.stem.rsplit("_", 1)[-1] + sc, "#e67e22" if bad else ("#2e8b57" if k is not None else "#999"),
                           "" if k is None else str(k.needs_review or "passed")))
    score = "-" if pd.isna(row.vlm_score) else f"{row.vlm_score:g}"
    if not crops and (row.feathers_saved or 0) > 0:
        crops = ['<i style="color:#888">crops not on Drive (slide left out of the share)</i>']
    return (_img(BEFORE_DIR / "overlays" / f"{stem}_BoundingBoxes.jpg", (300, 340), "width:300px")
            + f'<div><b>slide score {score}</b> · {int(row.feathers_saved or 0)} crops · '
            f'<span style="background:#eee;padding:1px 4px">bird_id {html.escape(str(row.bird_id))}</span> '
            f'<span style="background:#eee;padding:1px 4px">year {_year(row.date_text)}</span></div>'
            f'<div style="color:#555;font-size:11px">{html.escape(str(row.vlm_notes or ""))}</div>'
            f'<div>{"".join(crops)}</div>')

def _after_side(r):
    ov = on.find_overlay(AFTER_SEG, r.file_id, lx.result_date(r))
    chips = "".join(
        f'<span style="background:{STATUS_COLORS.get(f.status, "#eee")};padding:1px 4px;margin:1px;border-radius:3px;'
        f'display:inline-block" title="{html.escape(f.status + (": " + f.note if f.note else ""))}">'
        f'<b>{k}</b> {html.escape(lx.export_value(f.value, f.status) or "-")}</span>' for k, f in r.fields.items())
    crops = []
    for f in r.feathers:
        st, sc = f.get("segment_status", "segmented"), f.get("judge_score")
        tip = f'{f.get("feather_id") or f["seq"]}: {f.get("needs_review") or "passed"} {f.get("judge_notes") or ""}'
        if st == "segmented":
            color = "#c0392b" if f.get("duplicate_of") else ("#e67e22" if f.get("needs_review") else "#2e8b57")
            crops.append(_crop(AFTER_SEG / f["crop"], f'{f["seq"]} {"" if sc is None else f"{float(sc):g}"}', color, tip))
        elif st == "fragment":
            crops.append(_crop(AFTER_SEG / f["crop"], "frag", "#888", tip, dashed=True))
        else:
            crops.append(f'<div style="display:inline-block;width:40px;height:70px;margin:1px;border:3px dashed #c0392b;'
                         f'text-align:center;font-size:10px;vertical-align:top" title="{html.escape(tip)}">'
                         f'{"missed" if st == "possible_missed" else "count?"}</div>')
    ids = ", ".join(f["feather_id"] + (f' <span style="color:#c0392b">= {f["duplicate_of"]}</span>'
                                       if f.get("duplicate_of") else "") for f in lx.segmented(r.feathers))
    return (_img(ov, (300, 340), "width:300px") + f"<div>{chips}</div>"
            + f'<div style="font-size:11px"><b>IDs</b> {ids or "-"}</div><div>{"".join(crops)}</div>')

def _change(n, r):
    """(-1 better, 0 same, 1 worse, sort key): count vs the hand count first, then failing feathers."""
    want = (gold.get(n) or {}).get("feather_count", "").strip()
    nb, na = int(before.feathers_saved.get(n) or 0), len(lx.segmented(r.feathers))
    if want:
        eb, ea = abs(nb - int(want)), abs(na - int(want))
        if ea != eb:
            return (-1 if ea < eb else 1), ea - eb
    if bchk is not None and before_crops(Path(n).stem):
        # a feather counts as a problem if it fails a check (hidden left out on both sides) or has glue/card left
        fb = sum(str(getattr(bscore.get(p.name), "needs_review", "") or "") not in ("", "nan")
                 or glue_of.get(("before", p.name), 0) > 0 for p in before_crops(Path(n).stem))
        fa = sum(bool([i for i in str(f.get("needs_review") or "").split("; ") if i and not i.endswith("hidden")])
                 or glue_of.get(("after", f["crop"]), 0) > 0 for f in lx.segmented(r.feathers))
        if fa != fb:
            return (-1 if fa < fb else 1), fa - fb
    return 0, 0

def _card(n, r):
    ch, _ = _change(n, r)
    border = {-1: "#2e8b57", 0: "#bbb", 1: "#c0392b"}[ch]
    hand = (gold.get(n) or {}).get("feather_count", "").strip()
    return (f'<div style="border:4px solid {border};border-radius:6px;padding:6px;font-size:12px;font-family:sans-serif;'
            f'margin-bottom:10px"><div style="font-weight:bold">{html.escape(n)}'
            + (f' <span style="font-weight:normal;color:#555">(hand count {hand})</span>' if hand else "") + "</div>"
            f'<div style="display:flex;gap:14px"><div style="width:310px"><div style="color:#888">BEFORE</div>'
            f'{_before_side(n)}</div><div style="width:310px"><div style="color:#888">AFTER</div>{_after_side(r)}</div>'
            f"</div></div>")

pairs = [(r.file_id, r) for r in results if r.file_id in before.index and (not GRID_ONLY_GOLD or r.file_id in gold)]
if GRID_ORDER == "change":
    pairs.sort(key=lambda nr: (_change(*nr)[0] == 0, _change(*nr)[1]))
changes = Counter(_change(n, r)[0] for n, r in pairs)
display(Markdown(f"**{len(pairs)} slides**: {changes[-1]} better after (green), {changes[1]} worse (red), "
                 f"{changes[0]} no clear change (grey)"))
display(HTML('<div style="display:flex;flex-wrap:wrap;gap:10px">' + "".join(_card(n, r) for n, r in pairs) + "</div>"))

## 6. The after table for analysis
`feathers_usable.csv` in the run folder holds only the usable feathers: a feather ID, all checks passed, not a
duplicate. Below: the rows left out, and why.

In [ ]:
out = fdf[fdf.usable != "yes"]
print(f"{(fdf.usable == 'yes').sum()} usable of {len(fdf)} rows; left out by reason:")
reason = out.excluded_because.fillna("").str.replace(r"^duplicate of .*", "duplicate", regex=True).str.split(":").str[0]
display(reason.value_counts().to_frame("rows").T)
display(out[["file_id", "seq", "feather_id", "segment_status", "excluded_because", "judge_score", "crop"]])